# HOMEWORK 5

Done by: **Bohdan Pinchuk**

Link: https://github.com/BogdanPinchuk/ComputerVision-PBY_HW5

In [1]:
# # Silent installation or update
#
# # Clean cache
# !python3 -m pip cache purge -q
#
# # Force updating
# package_update = [
#     "pip",
#     "scikit-learn",
#     "pandas",
# ]
#
# for package_name in package_update:
#     !bash -c "python3 -m pip install -U '{package_name}' -q"
#
# # Install missing packages
# package_array = [
#     "jinja2",
#     "ipywidgets",
#     "nbformat",
#     "numpy",
#     "matplotlib",
#     "opencv-python",
# ]
#
# for package_name in package_array:
#     !bash -c "python3 -m pip show '{package_name}' > /dev/null 2>&1 || python3 -m pip install -U '{package_name}' -q"


In [1]:
# # Synchronization with remote source
#
# import shutil
# from pathlib import Path
#
# # Input images
# hm_version = 5
#
# # Solution
# git_project_url = f"https://github.com/BogdanPinchuk/ComputerVision-PBY_HW{hm_version}.git"
# main_file_name = f"Bohdan_Pinchuk_CV_HW{hm_version}.ipynb"
#
# # upload all files
# current_path = !pwd
# current_path = current_path[0]
# parent_path = !dirname "$current_path"
# parent_path = parent_path[0]
# temp_path = f"{parent_path}/temp"
#
# # Clone images
# !rm -rf "$temp_path"
# !git clone "$git_project_url" "$temp_path"
#
# source = Path(temp_path)
# destination = Path(current_path)
# exclude = {main_file_name, ".git", ".idea"}
#
# for item in source.iterdir():
#     if item.name in exclude:
#         continue
#
#     target = destination / item.name
#     if item.is_dir():
#         shutil.copytree(item, target, dirs_exist_ok=True)
#     else:
#         shutil.copy2(item, target)
#
# # Clean temp folder
# !rm -rf "$temp_path"

Cloning into '/Users/bohdanpinchuk/Documents/Data Science/Development/Computer Vision/Practical_tasks/temp'...
remote: Enumerating objects: 34, done.
remote: Counting objects: 100% (34/34), done.
remote: Compressing objects: 100% (28/28), done.
remote: Total 34 (delta 4), reused 34 (delta 4), pack-reused 0 (from 0)
Receiving objects: 100% (34/34), 7.08 MiB | 21.38 MiB/s, done.
Resolving deltas: 100% (4/4), done.



In this homework you are going to implement the **Floyd-Steinberg dithering** algorithm. Dithering, in general, means that we are adding noise to the signal (in our case digital image) in order to perceive it better. In other words, by adding the noise the objective quality will be worse but the subjective quality will be better (i.e. the image will "look" better).

The details of FS dithering can be found in this [wiki](https://en.wikipedia.org/wiki/Floyd%E2%80%93Steinberg_dithering) page. In order to implement the dithering, we will implement the following steps:
* Define colour palette
* Quantize the image to obtain the baseline and compute the average quantization error
* Implement FS dithering and compute the average quantization error

You will also have to answer the question at the end of this notebook.

Note: In this homework, you will have the chance to earn some extra points. See the "Bonus" section at the end of the notebook. Good luck!

As always, you are encouraged to use your own images :-)

In [3]:
from difflib import diff_bytes
# Load data

from pathlib import Path

import cv2

import apps.reporter as rpt

# Input data
# file_name = "resources/images/570494.jpg"
file_name = "resources/images/kodim23.png"
# file_name = "resources/images/kodim05.png"

# Solution
file_path = Path(file_name)

# Load an image (you can freely choose any image you like)
if file_path.exists():
    img_bgr = cv2.imread(file_path)
else:
    raise FileNotFoundError("File doesn't exist!")

# Convert it to RGB
if img_bgr is None:
    raise ValueError('Incorrect input data "img_bgr"!')

img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

rp = rpt.Reporter()
rp.add_item("(rows, cols, channels)", str(img_rgb.shape))

# Print results
rp.print_pd_report("Image shape")


FileNotFoundError: File doesn't exist!

Let's load the image.

In [ ]:
# Graphic results

import matplotlib.pyplot as plt

# Input data
data = img_rgb

# Solution
_, ax = plt.subplots(figsize=(9, 6))

ax.imshow(data)
ax.axis(False)

# Plot it
plt.title("Input picture")
plt.show()


Let's start with gray tones first.

In [ ]:
# Image Quantization - Baseline

import numpy as np

# Input data
img_rgb_float = img_rgb.copy().astype(np.float32)

# Black, dark gray, light gray, white
colors = np.array([[0, 0, 0],
                   [64, 64, 64],
                   [192, 192, 192],
                   [255, 255, 255]])

# Solution
colors_per_channel = colors[:, 0].copy()


# Both parts of the homework need the same operation: given a pixel, find the closest colour of the
# palette. Let's write it once as a small function and reuse it.
def closest_color(pixel: np.ndarray | float, color_palette: np.ndarray):
    # Compute the distance from the pixel to every colour of the palette
    # Hint: colors - pixel gives you all the differences at once
    if pixel.ndim == 0:
        distances = np.abs(pixel - color_palette)
    else:
        distances = np.linalg.norm(pixel - color_palette, axis=1)
    # Return the colour of the palette with the smallest distance
    idx = np.argmin(distances)

    return color_palette[idx]


# Using the colour palette, let's quantize the original image. This is our baseline: every pixel is
# replaced by the closest palette colour, independently of its neighbours.

# Prepare for quantization
rows, cols, channels = img_rgb_float.shape
quantized = np.zeros_like(img_rgb_float)
quantized_per_channel = quantized.copy()

# Apply quantization
for r in range(rows):
    for c in range(cols):
        # Extract the original pixel value
        pixel = img_rgb_float[r, c, :]

        # Find the closest colour from the palette
        new_pixel = closest_color(pixel, colors)
        # Apply quantization
        quantized[r, c, :] = new_pixel.copy()

        new_pixel = []
        for i in range(channels):
            new_pixel.append(closest_color(pixel[i], colors_per_channel))

        quantized_per_channel[r, c, :] = new_pixel.copy()

# Show quantized image (don't forget to cast back to uint8)
quantized = np.clip(quantized, 0, 255).astype(np.uint8)
quantized_per_channel = np.clip(quantized_per_channel, 0, 255).astype(np.uint8)

# Print results
images_dict_q = {
    "Original image": img_rgb,
    "Quantization in gray tones": quantized,
    "Quantization per channel": quantized_per_channel,
}

In [ ]:
# Graphic results

import math
import numpy as np
import matplotlib.pyplot as plt

# Input data
n_img_cols = 2
images_dict = images_dict_q
n_img_rows = math.ceil(len(images_dict) / n_img_cols)

# Solution
_, axes = plt.subplots(n_img_rows, n_img_cols, figsize=(12, 5 * n_img_rows))
axes_list = np.array(axes).flatten()

for idx, (img_name, img_data) in enumerate(images_dict.items()):
    ax = axes_list[idx]
    ax.set_title(img_name, pad=10, loc='center', color='black')
    if len(img_data.shape) == 3:
        ax.imshow(img_data)
    else:
        ax.imshow(img_data, cmap='gray')
    ax.axis(False)

# remove empty charts
for idx in range(len(images_dict), len(axes_list)):
    axes_list[idx].remove()

# Plot
plt.suptitle("Image Quantization")
plt.tight_layout()
plt.show()


To be able to compare the two approaches we need a number. Let's use the **average quantization error**,
i.e. the mean absolute difference between the original and the quantized image, averaged over all pixels
and all three channels.

In [ ]:
# Compute average squared quantization error

import apps.main as mn
from IPython.core.display import Markdown

# Input data
# avg_quant_error =

# Solution

# Print results
display(Markdown(f"## Estimate: Quantization in gray tones"))
df_q_gt = mn.calc_metrics(quantized, img_rgb, 255)
display(Markdown(f"## Estimate: Quantization per channel"))
df_q_pc = mn.calc_metrics(quantized_per_channel, img_rgb, 255)


#### Floyd-Steinberg Dithering

We are now going to implement the FS dithering and compare it to the optimally quantized image we have
calculated above.

The idea is a **feedback loop**: after quantizing a pixel we know exactly how much we got it wrong,

$$ \varepsilon = p_{orig} - p_{quant} $$

and we push that error onto the neighbours that have not been processed yet. If a pixel was made too
dark, its neighbours are made brighter on purpose, so that the error averages out to zero over a small
area. The error is distributed according to the FS diffusion matrix (`*` marks the current pixel, the
image is scanned left to right, top to bottom)

$$ \frac{1}{16}
\begin{bmatrix}
 - & * & 7 \\
 3 & 5 & 1
\end{bmatrix} $$

so the pixel to the right receives 7/16 of the error, the one below-left 3/16, the one below 5/16 and
the one below-right 1/16.

Two practical notes:
* The diffused error has to be accumulated in a **copy** of the image (`img_tmp`). Each pixel must be
  quantized using the value that already contains the error of its neighbours, not the original value.
* We skip the first and the last row and column so that we do not have to handle the borders, which is
  why the loop starts at 1.

In [ ]:
# Image Dithering - Floyd-Steinberg dithering algorithm

# Input data
# Make a temporal copy of the original image, we will need it for error diffusion
img_tmp = img_rgb_float.copy()
dithering = img_rgb_float.copy()
dithering_per_channel = img_rgb_float.copy()

# Solution
for r in range(1, rows - 1):
    for c in range(1, cols - 1):
        # Extract the pixel value, including the error diffused from the neighbours
        pixel = dithering[r, c, :]

        # Find the closest colour from the palette
        new_pixel = closest_color(pixel, colors)

        # Compute quantization error
        quant_error = pixel - new_pixel

        # Diffuse the quantization error according to the FS diffusion matrix
        # Note: You need more than one line of code here
        dithering[r, c + 1, :] += quant_error * (7.0 / 16.0)
        dithering[r + 1, c - 1, :] += quant_error * (3.0 / 16.0)
        dithering[r + 1, c, :] += quant_error * (5.0 / 16.0)
        dithering[r + 1, c + 1, :] += quant_error * (1.0 / 16.0)

        # Apply dithering
        dithering[r, c, :] = new_pixel

        pixel = dithering_per_channel[r, c, :]
        new_pixel = []
        for i in range(channels):
            new_pixel.append(closest_color(pixel[i], colors_per_channel))

        # Compute quantization error
        quant_error = pixel - new_pixel

        # Diffuse the quantization error according to the FS diffusion matrix
        # Note: You need more than one line of code here
        dithering_per_channel[r, c + 1, :] += quant_error * (7.0 / 16.0)
        dithering_per_channel[r + 1, c - 1, :] += quant_error * (3.0 / 16.0)
        dithering_per_channel[r + 1, c, :] += quant_error * (5.0 / 16.0)
        dithering_per_channel[r + 1, c + 1, :] += quant_error * (1.0 / 16.0)
        
        # Apply dithering
        dithering_per_channel[r, c, :] = new_pixel

# Show quantized image (don't forget to cast back to uint8)
dithering = np.clip(dithering, 0, 255).astype(np.uint8)
dithering_per_channel = np.clip(dithering_per_channel, 0, 255).astype(np.uint8)

# Print results
images_dict_d = {
    "Original image": img_rgb,
    "Dithering in gray tones": dithering,
    "Dithering per channel": dithering_per_channel,
}



In [ ]:
# Graphic results

import math
import numpy as np
import matplotlib.pyplot as plt

# Input data
images_dict = images_dict_d
n_img_cols = 2
n_img_rows = math.ceil(len(images_dict) / n_img_cols)

# Solution
_, axes = plt.subplots(n_img_rows, n_img_cols, figsize=(12, 5 * n_img_rows))
axes_list = np.array(axes).flatten()

for idx, (img_name, img_data) in enumerate(images_dict.items()):
    ax = axes_list[idx]
    ax.set_title(img_name, pad=10, loc='center', color='black')
    if len(img_data.shape) == 3:
        ax.imshow(img_data)
    else:
        ax.imshow(img_data, cmap='gray')
    ax.axis(False)

# remove empty charts
for idx in range(len(images_dict), len(axes_list)):
    axes_list[idx].remove()

# Plot
plt.suptitle("Image Dithering")
plt.tight_layout()
plt.show()


In [ ]:
# Compute average squared quantization error for dithered image

import apps.main as mn
from IPython.core.display import Markdown

# Input data
# avg_dith_error =

# Solution

# Print results
display(Markdown(f"## Estimate: Dithering in gray tones"))
df_d_gt = mn.calc_metrics(dithering, img_rgb, 255)
display(Markdown(f"## Estimate: Dithering per channel"))
df_d_pc = mn.calc_metrics(dithering_per_channel, img_rgb, 255)

### Questions
* Which image has higher quantization error? Optimally quantized or dithered?
* Which image looks better to you?
* Can you repeat the same process using only two colours: black and white? Show me :-)

### Bonus Points

Repeat the homework using a different image palette. For instance, you can use an optimal colour
palette that we can calculate via k-means algorithm. The following snippet of code will give you the 16
optimal colours for your original image.

In [ ]:
from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=16).fit(np.reshape(img, (-1, 3)))
colors = kmeans.cluster_centers_

Apply FS dithering the same way you did before.
* How does the result look like to you?
* What happens if we use 32 colours?
* And what happens if we use 256 colours?